# Day 4 — Gold: `care_gap_a1c`, prototyped

One row per diabetic patient. The product of the whole pipeline, sourced from
Silver only. Written here first, reviewed, then ported to `pipeline/build_gold.py`.

**Run order:** `load_bronze.py` → `corrupt.py` → `validate.py` → this notebook.

**DuckDB allows one process on the file at a time.** While this kernel holds `con`,
the `.py` scripts cannot open the database. Run the last cell (`con.close()`)
before running any `.py`, or shut the kernel down. Closing the tab does not stop it.

In [1]:
import duckdb
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 50)

from cohort import DIABETES_CODES          # the eight SNOMED codes, Decision D5

DB   = "../data/warehouse/clinical.duckdb"
RAW  = "../data/raw/csv"
ASOF = "2026-08-23"                       # Decision D7
A1C  = "4548-4"
GAP_DAYS = 365
DX = ", ".join(f"'{c}'" for c in DIABETES_CODES)

con = duckdb.connect(DB)
def q(sql): return con.sql(sql).df()
print(len(DIABETES_CODES), "cohort codes |", "as of", ASOF)

8 cohort codes | as of 2026-08-23


## 4.1 — Lock both definitions (blocking)

**D5 — Diabetic patient.** Any of the eight SNOMED codes ever recorded on a patient
who is **alive on the as-of date**.

- *Ever recorded, not "unresolved":* 0 of 835 diabetes condition rows carry a stop
  date (Day 1), so "unresolved" and "ever" are the same set here. On real data,
  where type 2 diabetes is still managed rather than cured, I would keep "ever" and
  treat a resolved diabetes row as a data-quality question, not a cohort exit.
- *Alive on the as-of date:* a care-gap list is a call list. 45 of the 161 people
  carrying a diabetes code died before 2026-08-23. HEDIS excludes deceased members
  from the denominator; so do we.

**D6 — Open A1c gap.** No A1c result (LOINC `4548-4`, numeric value, observed on or
before the as-of date) in the 365 days before the as-of date. A patient who has
**never** been tested is a gap. Exactly 365 days is not a gap; 366 is.

- *Ordered but not resulted:* not observable here — Synthea has no orders table and
  no procedure row mentions A1c. On real data "we ordered it and the patient never
  went" and "we never ordered it" are different problems with different fixes, and
  a report counting only results conflates them. v1 conflates them and says so.
- *Results in another lab system:* also invisible. The number is "no result *we can
  see*", which is what every care-gap report actually measures.
- *Remediated values count:* the 20 A1cs corrected from 250 to 10.3 are results
  (Decision D4). A reviewer who disagrees excludes `_dq_status = 'remediated'`.

In [2]:
q(f"""
SELECT 'any of 8 codes ever recorded'            AS definition, count(DISTINCT c.patient_id) AS patients
FROM silver_conditions c WHERE c.snomed_code IN ({DX})
UNION ALL
SELECT '... and alive on {ASOF} (D5, the denominator)', count(DISTINCT c.patient_id)
FROM silver_conditions c JOIN silver_patients p USING (patient_id)
WHERE c.snomed_code IN ({DX}) AND (p.death_date IS NULL OR p.death_date > DATE '{ASOF}')
UNION ALL
SELECT '... deceased before as-of (excluded)', count(DISTINCT c.patient_id)
FROM silver_conditions c JOIN silver_patients p USING (patient_id)
WHERE c.snomed_code IN ({DX}) AND p.death_date <= DATE '{ASOF}'
""")

,definition,patients
0,any of 8 codes ever recorded,161
1,"... and alive on 2026-08-23 (D5, the denominator)",116
2,... deceased before as-of (excluded),45


## 4.2 — Build `care_gap_a1c`

Three CTEs: the cohort, each patient's latest A1c, each patient's active
medication count. Then **one LEFT JOIN** from cohort to latest A1c. That join
direction is the whole of task 4.3 — see the next section for what an inner
join would silently do.

In [3]:
con.sql(f"""
CREATE OR REPLACE TABLE care_gap_a1c AS
WITH cohort AS (                                       -- D5
    SELECT DISTINCT c.patient_id, p.birth_date
    FROM silver_conditions c
    JOIN silver_patients p USING (patient_id)
    WHERE c.snomed_code IN ({DX})
      AND (p.death_date IS NULL OR p.death_date > DATE '{ASOF}')
),
latest_a1c AS (                                        -- newest numeric result on or before as-of
    SELECT patient_id, observed_at::DATE AS last_a1c_date, value AS last_a1c_value
    FROM silver_observations
    WHERE loinc_code = '{A1C}' AND value IS NOT NULL AND observed_at <= TIMESTAMP '{ASOF}'
    QUALIFY row_number() OVER (PARTITION BY patient_id ORDER BY observed_at DESC) = 1
),
active_meds AS (                                       -- active on the as-of date
    SELECT patient_id, count(*) AS active_med_count
    FROM silver_medications
    WHERE start_date <= DATE '{ASOF}' AND (end_date IS NULL OR end_date > DATE '{ASOF}')
    GROUP BY patient_id
)
SELECT c.patient_id,
       date_part('year', age(DATE '{ASOF}', c.birth_date))::INT       AS age,
       a.last_a1c_date,
       a.last_a1c_value,
       date_diff('day', a.last_a1c_date, DATE '{ASOF}')                AS days_since_a1c,   -- NULL when never tested
       (a.last_a1c_date IS NULL OR date_diff('day', a.last_a1c_date, DATE '{ASOF}') > {GAP_DAYS}) AS gap_flag,
       coalesce(m.active_med_count, 0)                                 AS active_med_count,
       DATE '{ASOF}'                                                   AS asof_date
FROM cohort c
LEFT JOIN latest_a1c  a USING (patient_id)                            -- LEFT: never-tested patients must survive
LEFT JOIN active_meds m USING (patient_id)
""")
q("SELECT * FROM care_gap_a1c ORDER BY gap_flag DESC, days_since_a1c DESC NULLS FIRST LIMIT 8")

,patient_id,age,last_a1c_date,last_a1c_value,days_since_a1c,gap_flag,active_med_count,asof_date
0,f5241669-2b5e-fce5-d5ba-b359ceeed448,61,NaT,NaN,<NA>,True,6,2026-08-23
1,23018670-880f-61c0-1ede-08c641bd8f84,79,NaT,NaN,<NA>,True,12,2026-08-23
2,dcbb4855-fa56-b26c-c62b-2b7fdebc8e05,57,NaT,NaN,<NA>,True,7,2026-08-23
3,dd31b260-5ad0-2bff-3218-031136ec287c,85,NaT,NaN,<NA>,True,4,2026-08-23
4,ba2fbe06-ef00-35df-8d54-1a3fe891c327,54,NaT,NaN,<NA>,True,6,2026-08-23
5,f52c7b09-9aa3-c2b6-2891-9422737ce2e2,63,NaT,NaN,<NA>,True,6,2026-08-23
6,8670ef45-e53b-be2d-3cb7-2fb40a2343ce,73,NaT,NaN,<NA>,True,6,2026-08-23
7,f7aa1b72-e259-1d2e-a9f7-8adbb68fa386,56,NaT,NaN,<NA>,True,3,2026-08-23


In [4]:
q("""
SELECT count(*)                                         AS cohort,
       count(*) FILTER (WHERE gap_flag)                 AS open_gaps,
       round(100.0 * count(*) FILTER (WHERE gap_flag) / count(*), 1) AS gap_rate_pct,
       count(*) FILTER (WHERE last_a1c_date IS NULL)    AS never_tested,
       min(age) AS min_age, round(avg(age)) AS avg_age, max(age) AS max_age
FROM care_gap_a1c
""")

,cohort,open_gaps,gap_rate_pct,never_tested,min_age,avg_age,max_age
0,116,25,21.6,21,36,65.0,103


## 4.3 — The never-tested patient, and the inner-join bug

The highest-risk person on a care-gap list is the diabetic with no A1c on record.
An inner join from cohort to observations deletes exactly those people: the query
runs, the numbers look plausible, and the patients who most need a call are gone.
Here is the same build with the one word changed:

In [5]:
q(f"""
WITH cohort AS (
    SELECT DISTINCT c.patient_id FROM silver_conditions c JOIN silver_patients p USING (patient_id)
    WHERE c.snomed_code IN ({DX}) AND (p.death_date IS NULL OR p.death_date > DATE '{ASOF}')),
latest AS (
    SELECT patient_id, max(observed_at) AS last_a1c FROM silver_observations
    WHERE loinc_code = '{A1C}' AND value IS NOT NULL GROUP BY 1)
SELECT 'LEFT JOIN  (correct)' AS build, count(*) AS rows, count(*) FILTER (WHERE last_a1c IS NULL) AS never_tested_kept
FROM cohort LEFT JOIN latest USING (patient_id)
UNION ALL
SELECT 'INNER JOIN (the bug)', count(*), count(*) FILTER (WHERE last_a1c IS NULL)
FROM cohort JOIN latest USING (patient_id)
""")

,build,rows,never_tested_kept
0,LEFT JOIN (correct),116,21
1,INNER JOIN (the bug),95,0


The inner-join version reports fewer patients, zero of them never-tested, and
nothing errors. `days_since_a1c` is NULL for those patients — null arithmetic
propagates — which is why `gap_flag` tests `last_a1c_date IS NULL` explicitly
rather than relying on `NULL > 365`, which is *unknown*, not true.

## 4.4 — Hand-verify three patients, raw CSV to Gold row

One with a recent A1c (not flagged), one whose last A1c is over a year old
(flagged), one never tested (flagged, null date). Each traced from the raw
Synthea files, not from Silver — so the whole pipeline is checked, not just the
last step.

In [6]:
picks = q("""
(SELECT 'recent'  AS kind, patient_id FROM care_gap_a1c WHERE NOT gap_flag ORDER BY md5(patient_id) LIMIT 1)
UNION ALL
(SELECT 'stale',  patient_id FROM care_gap_a1c WHERE gap_flag AND last_a1c_date IS NOT NULL ORDER BY md5(patient_id) LIMIT 1)
UNION ALL
(SELECT 'never',  patient_id FROM care_gap_a1c WHERE last_a1c_date IS NULL ORDER BY md5(patient_id) LIMIT 1)
""")
for kind, pid in picks.itertuples(index=False):
    print(f"\n=== {kind.upper()}  {pid} ===")
    print("raw patients.csv   :", q(f"SELECT BIRTHDATE, DEATHDATE, FIRST, LAST FROM read_csv_auto('{RAW}/patients.csv', all_varchar=true) WHERE Id = '{pid}'").to_dict('records'))
    print("raw conditions.csv :", q(f"SELECT CODE, DESCRIPTION, START FROM read_csv_auto('{RAW}/conditions.csv', all_varchar=true) WHERE PATIENT = '{pid}' AND CODE IN ({DX}) ORDER BY START LIMIT 3").to_dict('records'))
    a1c = q(f"SELECT DATE, VALUE, UNITS FROM read_csv_auto('{RAW}/observations.csv', all_varchar=true) WHERE PATIENT = '{pid}' AND CODE = '{A1C}' ORDER BY DATE DESC LIMIT 3")
    print("raw observations.csv (A1c, newest first):", a1c.to_dict('records') if len(a1c) else "NONE - never tested")
    print("Gold row           :", q(f"SELECT age, last_a1c_date, last_a1c_value, days_since_a1c, gap_flag, active_med_count FROM care_gap_a1c WHERE patient_id = '{pid}'").to_dict('records'))


=== RECENT  c24941da-8187-d66c-bfc0-926199a69c9f ===


raw patients.csv   : [{'BIRTHDATE': '1977-05-03', 'DEATHDATE': None, 'FIRST': 'Augustine565', 'LAST': 'Nicolas769'}]


raw conditions.csv : [{'CODE': '44054006', 'DESCRIPTION': 'Diabetes mellitus type 2 (disorder)', 'START': '2008-07-15'}, {'CODE': '127013003', 'DESCRIPTION': 'Disorder of kidney due to diabetes mellitus (disorder)', 'START': '2016-11-01'}, {'CODE': '90781000119102', 'DESCRIPTION': 'Microalbuminuria due to type 2 diabetes mellitus (disorder)', 'START': '2022-12-13'}]


raw observations.csv (A1c, newest first): [{'DATE': '2026-06-16T07:41:50Z', 'VALUE': '4.5', 'UNITS': '%'}, {'DATE': '2026-04-14T07:41:50Z', 'VALUE': '4.5', 'UNITS': '%'}, {'DATE': '2026-03-17T07:41:50Z', 'VALUE': '4.5', 'UNITS': '%'}]
Gold row           : [{'age': 49, 'last_a1c_date': Timestamp('2026-06-16 00:00:00'), 'last_a1c_value': 4.5, 'days_since_a1c': 68, 'gap_flag': False, 'active_med_count': 5}]

=== STALE  df8d2de6-32a1-262d-b07f-1b73e77efef8 ===


raw patients.csv   : [{'BIRTHDATE': '1977-01-27', 'DEATHDATE': None, 'FIRST': 'Erin498', 'LAST': 'Collier206'}]


raw conditions.csv : [{'CODE': '44054006', 'DESCRIPTION': 'Diabetes mellitus type 2 (disorder)', 'START': '2011-04-14'}, {'CODE': '368581000119106', 'DESCRIPTION': 'Neuropathy due to type 2 diabetes mellitus (disorder)', 'START': '2017-02-02'}]


raw observations.csv (A1c, newest first): [{'DATE': '2025-02-20T08:40:36Z', 'VALUE': '7.0', 'UNITS': '%'}, {'DATE': '2024-01-18T08:40:36Z', 'VALUE': '7.0', 'UNITS': '%'}, {'DATE': '2023-02-16T08:40:36Z', 'VALUE': '7.0', 'UNITS': '%'}]
Gold row           : [{'age': 49, 'last_a1c_date': Timestamp('2025-02-20 00:00:00'), 'last_a1c_value': 7.0, 'days_since_a1c': 549, 'gap_flag': True, 'active_med_count': 3}]

=== NEVER  f52c7b09-9aa3-c2b6-2891-9422737ce2e2 ===


raw patients.csv   : [{'BIRTHDATE': '1962-12-26', 'DEATHDATE': None, 'FIRST': 'Samual455', 'LAST': 'Doyle959'}]


raw conditions.csv : [{'CODE': '127013003', 'DESCRIPTION': 'Disorder of kidney due to diabetes mellitus (disorder)', 'START': '2011-12-28'}, {'CODE': '90781000119102', 'DESCRIPTION': 'Microalbuminuria due to type 2 diabetes mellitus (disorder)', 'START': '2012-07-25'}, {'CODE': '157141000119108', 'DESCRIPTION': 'Proteinuria due to type 2 diabetes mellitus (disorder)', 'START': '2023-05-31'}]


raw observations.csv (A1c, newest first): NONE - never tested
Gold row           : [{'age': 63, 'last_a1c_date': NaT, 'last_a1c_value': nan, 'days_since_a1c': None, 'gap_flag': True, 'active_med_count': 6}]


## V4.x — validations

In [7]:
checks = {
 "V4.1 grain: rows == distinct patients":
    q("SELECT count(*) = count(DISTINCT patient_id) AS ok FROM care_gap_a1c").ok[0],
 "V4.2 never-tested survived, all flagged":
    q("SELECT count(*) > 0 AND bool_and(gap_flag) AS ok FROM care_gap_a1c WHERE last_a1c_date IS NULL").ok[0],
 "V4.3 days_since null-safe, non-negative, < 40000":
    q("SELECT count(*) = 0 AS ok FROM care_gap_a1c WHERE (last_a1c_date IS NULL AND days_since_a1c IS NOT NULL) OR days_since_a1c < 0 OR days_since_a1c > 40000").ok[0],
 "V4.4 gap_flag matches definition (>365)":
    q(f"SELECT bool_and(CASE WHEN gap_flag THEN days_since_a1c IS NULL OR days_since_a1c > {GAP_DAYS} ELSE days_since_a1c <= {GAP_DAYS} END) AS ok FROM care_gap_a1c").ok[0],
 "V4.5 cohort == written D5 definition":
    q(f"""SELECT (SELECT count(*) FROM care_gap_a1c) = (SELECT count(DISTINCT c.patient_id) FROM silver_conditions c JOIN silver_patients p USING (patient_id)
          WHERE c.snomed_code IN ({DX}) AND (p.death_date IS NULL OR p.death_date > DATE '{ASOF}')) AS ok""").ok[0],
 "V4.6 no quarantined patient in Gold":
    q("SELECT count(*) = 0 AS ok FROM care_gap_a1c g JOIN quarantine x ON x.source_row_id = g.patient_id AND x.source_table = 'bronze_patients'").ok[0],
 "V4.8 age plausible (0..120)":
    q("SELECT min(age) >= 0 AND max(age) <= 120 AS ok FROM care_gap_a1c").ok[0],
 "V4.9 gap rate believable (not 0%, not 100%)":
    q("SELECT avg(gap_flag::INT) BETWEEN 0.01 AND 0.99 AS ok FROM care_gap_a1c").ok[0],
}
for k, v in checks.items(): print(f"  {'ok ' if v else 'FAIL'}  {k}")
print("\nV4.4 boundary detail:")
display(q("SELECT gap_flag, count(*) n, min(days_since_a1c) min_days, max(days_since_a1c) max_days FROM care_gap_a1c GROUP BY 1 ORDER BY 1"))
print("V4.8 age distribution:")
display(q("SELECT (age // 10) * 10 AS decade, count(*) n, count(*) FILTER (WHERE gap_flag) gaps FROM care_gap_a1c GROUP BY 1 ORDER BY 1"))

  ok   V4.1 grain: rows == distinct patients
  ok   V4.2 never-tested survived, all flagged
  ok   V4.3 days_since null-safe, non-negative, < 40000
  ok   V4.4 gap_flag matches definition (>365)
  ok   V4.5 cohort == written D5 definition
  ok   V4.6 no quarantined patient in Gold
  ok   V4.8 age plausible (0..120)
  ok   V4.9 gap rate believable (not 0%, not 100%)

V4.4 boundary detail:


,gap_flag,n,min_days,max_days
0,False,91,3,357
1,True,25,540,2540


V4.8 age distribution:


,decade,n,gaps
0,30,2,1
1,40,10,2
2,50,26,6
3,60,43,10
4,70,22,5
5,80,5,1
6,90,6,0
7,100,2,0


## Ready to move to `build_gold.py`

Port as-is: the three CTEs and the LEFT JOIN, `asof_date` on every row, the V4.x
checks as assertions. Then `run_all.py` for task 4.5 — one command from a clean
warehouse to Gold.

Record: **D5** now includes "alive on the as-of date" (denominator 116, not 161);
**D6** as written above. Both go into `DATA_DICTIONARY.md` and the Decisions sheet.

## Release the database

Always the last cell. Frees the file lock so the `.py` scripts can run.

In [8]:
con.close()
print("connection closed - the .py scripts can run now")

connection closed - the .py scripts can run now
